In [2]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine('sqlite:///../data/olist.db')

orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')

print("Orders:", orders.shape)
print("Payments:", payments.shape)
print("Customers:", customers.shape)

Orders: (99441, 8)
Payments: (103886, 5)
Customers: (99441, 5)


In [3]:
orders.to_sql('orders', engine, if_exists='replace', index=False)
payments.to_sql('payments', engine, if_exists='replace', index=False)
customers.to_sql('customers', engine, if_exists='replace', index=False)

print("Loaded to database.")

Loaded to database.


In [4]:
print(pd.read_sql("SELECT COUNT(*) AS n FROM orders", engine))
print(pd.read_sql("SELECT COUNT(*) AS n FROM payments", engine))
print(pd.read_sql("SELECT COUNT(*) AS n FROM customers", engine))

       n
0  99441
        n
0  103886
       n
0  99441


In [6]:
# 1. What order statuses exist? We only want 'delivered' orders for revenue analysis
print(pd.read_sql("SELECT order_status, COUNT(*) AS n FROM orders GROUP BY order_status", engine))

print()

# 2. Any missing purchase timestamps?
print(pd.read_sql("SELECT COUNT(*) AS missing_dates FROM orders WHERE order_purchase_timestamp IS NULL", engine))

print()

# 3. The important one — check for the customer_id vs customer_unique_id trap
print(pd.read_sql("""
    SELECT COUNT(DISTINCT customer_id) AS order_level_ids,
           COUNT(DISTINCT customer_unique_id) AS real_customers
    FROM customers
""", engine))

  order_status      n
0     approved      2
1     canceled    625
2      created      5
3    delivered  96478
4     invoiced    314
5   processing    301
6      shipped   1107
7  unavailable    609

   missing_dates
0              0

   order_level_ids  real_customers
0            99441           96096


In [8]:
customer_orders = pd.read_sql("""
    SELECT
        c.customer_unique_id,
        o.order_id,
        o.order_purchase_timestamp,
        p.payment_value
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    JOIN payments p ON o.order_id = p.order_id
    WHERE o.order_status = 'delivered'
""", engine)

print(customer_orders.shape)
customer_orders.head()

(100756, 4)


,customer_unique_id,order_id,order_purchase_timestamp,payment_value
0,7c396fd4830fd04220f754e42b4e5bff,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2.00
1,7c396fd4830fd04220f754e42b4e5bff,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,18.12
2,7c396fd4830fd04220f754e42b4e5bff,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,18.59
3,af07308b275d755c9edb36a90c618231,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,141.46
4,3a653a41f6f9fc3d2a113cf8398680e8,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,179.12


In [15]:
import pandas as pd

# Set "today" as one day after the most recent order in the dataset
snapshot_date = pd.to_datetime(customer_orders['order_purchase_timestamp']).max() + pd.Timedelta(days=1)

rfm = customer_orders.groupby('customer_unique_id').agg(
    recency_days=('order_purchase_timestamp', lambda x: (snapshot_date - pd.to_datetime(x).max()).days),
    frequency=('order_id', 'nunique'),
    monetary=('payment_value', 'sum')
).reset_index()

print(rfm.shape)
rfm.describe()

(93357, 4)


,recency_days,frequency,monetary
count,93357.000000,93357.000000,93357.000000
mean,237.936673,1.033420,165.198772
std,152.584315,0.209099,226.314579
min,1.000000,1.000000,9.590000
25%,114.000000,1.000000,63.060000
50%,219.000000,1.000000,107.780000
75%,346.000000,1.000000,182.560000
max,695.000000,15.000000,13664.080000


In [17]:
rfm['r_score'] = pd.qcut(rfm['recency_days'], 5, labels=[5,4,3,2,1]).astype(int)
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary'], 5, labels=[1,2,3,4,5]).astype(int)

rfm['rfm_score'] = rfm['r_score'].astype(str) + rfm['f_score'].astype(str) + rfm['m_score'].astype(str)

rfm.head(10)

,customer_unique_id,recency_days,frequency,monetary,r_score,f_score,m_score,rfm_score
0,0000366f3b9a7992bf8c76cfdf3221e2,112,1,141.90,4,1,4,414
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,27.19,4,1,1,411
2,0000f46a3911fa3c0805444483337064,537,1,86.22,1,1,2,112
3,0000f6ccb0745a6a4b88665a16c9f078,321,1,43.62,2,1,1,211
4,0004aac84e0df4da2b147fca70cf8255,288,1,196.89,2,1,4,214
5,0004bd2a26a76fe21f786e4fbd80607f,146,1,166.98,4,1,4,414
6,00050ab1314c0e55a6ca13cf7181fecf,132,1,35.38,4,1,1,411
7,00053a61a98854899e70ed204dd4bafe,183,1,419.18,3,1,5,315
8,0005e1862207bf6ccc02e4228effd9a0,543,1,150.12,1,1,4,114
9,0005ef4cd20d2893f0d9fbd94d3c0d97,170,1,129.76,4,1,3,413


In [18]:
rfm['r_score'] = pd.qcut(rfm['recency_days'], 5, labels=[5,4,3,2,1]).astype(int)
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['m_score'] = pd.qcut(rfm['monetary'], 5, labels=[1,2,3,4,5]).astype(int)

rfm['rfm_score'] = rfm['r_score'].astype(str) + rfm['f_score'].astype(str) + rfm['m_score'].astype(str)

rfm.head(10)

,customer_unique_id,recency_days,frequency,monetary,r_score,f_score,m_score,rfm_score
0,0000366f3b9a7992bf8c76cfdf3221e2,112,1,141.90,4,1,4,414
1,0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,27.19,4,1,1,411
2,0000f46a3911fa3c0805444483337064,537,1,86.22,1,1,2,112
3,0000f6ccb0745a6a4b88665a16c9f078,321,1,43.62,2,1,1,211
4,0004aac84e0df4da2b147fca70cf8255,288,1,196.89,2,1,4,214
5,0004bd2a26a76fe21f786e4fbd80607f,146,1,166.98,4,1,4,414
6,00050ab1314c0e55a6ca13cf7181fecf,132,1,35.38,4,1,1,411
7,00053a61a98854899e70ed204dd4bafe,183,1,419.18,3,1,5,315
8,0005e1862207bf6ccc02e4228effd9a0,543,1,150.12,1,1,4,114
9,0005ef4cd20d2893f0d9fbd94d3c0d97,170,1,129.76,4,1,3,413


In [19]:
def segment_customer(row):
    r, f, m = row['r_score'], row['f_score'], row['m_score']
    if r >= 4 and f >= 4:
        return 'Champions'
    elif r >= 4 and f <= 2:
        return 'New Customers'
    elif r <= 2 and f >= 4:
        return 'At Risk (High Value)'
    elif r <= 2 and f <= 2:
        return 'Hibernating'
    else:
        return 'Needs Attention'

rfm['segment'] = rfm.apply(segment_customer, axis=1)

rfm['segment'].value_counts()

segment
Needs Attention         33623
Hibernating             14986
New Customers           14984
Champions               14961
At Risk (High Value)    14803
Name: count, dtype: int64

In [20]:
segment_summary = rfm.groupby('segment').agg(
    customers=('customer_unique_id', 'count'),
    avg_monetary=('monetary', 'mean'),
    total_revenue=('monetary', 'sum')
).sort_values('total_revenue', ascending=False)

segment_summary['pct_of_customers'] = (segment_summary['customers'] / segment_summary['customers'].sum() * 100).round(1)
segment_summary['pct_of_revenue'] = (segment_summary['total_revenue'] / segment_summary['total_revenue'].sum() * 100).round(1)

segment_summary

,customers,avg_monetary,total_revenue,pct_of_customers,pct_of_revenue
segment,,,,,
Needs Attention,33623,159.780760,5372308.49,36.0,34.8
Champions,14961,176.956439,2647445.28,16.0,17.2
At Risk (High Value),14803,169.679879,2511771.25,15.9,16.3
New Customers,14984,163.427972,2448804.73,16.1,15.9
Hibernating,14986,162.960898,2442132.02,16.1,15.8


In [21]:
at_risk_revenue = rfm[rfm['segment'] == 'At Risk (High Value)']['monetary'].sum()
total_revenue = rfm['monetary'].sum()
pct_at_risk = at_risk_revenue / total_revenue * 100

print(f"Revenue at risk: R$ {at_risk_revenue:,.2f}")
print(f"Total revenue: R$ {total_revenue:,.2f}")
print(f"% of total revenue at risk: {pct_at_risk:.1f}%")

Revenue at risk: R$ 2,511,771.25
Total revenue: R$ 15,422,461.77
% of total revenue at risk: 16.3%


In [22]:
segment_summary = rfm.groupby('segment').agg(
    customers=('customer_unique_id', 'count'),
    avg_monetary=('monetary', 'mean'),
    total_revenue=('monetary', 'sum')
).sort_values('total_revenue', ascending=False)

segment_summary['pct_of_customers'] = (segment_summary['customers'] / segment_summary['customers'].sum() * 100).round(1)
segment_summary['pct_of_revenue'] = (segment_summary['total_revenue'] / segment_summary['total_revenue'].sum() * 100).round(1)

segment_summary

,customers,avg_monetary,total_revenue,pct_of_customers,pct_of_revenue
segment,,,,,
Needs Attention,33623,159.780760,5372308.49,36.0,34.8
Champions,14961,176.956439,2647445.28,16.0,17.2
At Risk (High Value),14803,169.679879,2511771.25,15.9,16.3
New Customers,14984,163.427972,2448804.73,16.1,15.9
Hibernating,14986,162.960898,2442132.02,16.1,15.8


In [23]:
rfm.to_csv('../data/processed/rfm_segments.csv', index=False)
print("Exported successfully.")
print(rfm.shape)

Exported successfully.
(93357, 9)
